In [ ]:
# all import at once, cuz why not :)
import pandas as pd
import numpy as np # for random data generation
import matplotlib.pyplot as plt
import kagglehub
import os
import torch
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, LabelEncoder #import OneHotEncoder
from sklearn.model_selection import KFold ,StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from tqdm import tqdm
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from torch.utils.data import TensorDataset, DataLoader
from torchvision.datasets import MNIST
from torch.optim import Adam, AdamW
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from collections import Counter





from IPython.display import clear_output
%pip install kagglehub catboost lightgbm tqdm -q
%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()
from catboost import CatBoostRegressor
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

print("Missing values Before:", df.isnull().sum().sum())

for col in df.columns:
    df[col] = df[col].fillna(df[col].mean())

print("Missing values after:", df.isnull().sum().sum())


In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df.info()
# no catagurical

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.columns  # DON'T SCALE THE TARGET
print(type(df))
print(numerical_cols)
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:

In [ ]:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# The softmax function
def softmax(z):
  z_shifted = z
  exp_z = np.exp(z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [ ]:
# Categorical cross entropy loss function using NumPy
def categorical_cross_entropy(y, y_hat):
  epsilon = 1e-15
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon)

  loss = -np.mean(np.sum(y * np.log(y_hat), axis=1))
  return loss

In [ ]:
def one_hot_encode(y, num_classes):
    y = np.array(y)
    m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    for i in range(m):
        # Identify which class this sample belongs to
        class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        one_hot[i, class_label] = 1

    return one_hot

In [ ]:
def gradient_descent(X, y, num_classes, lr, n_iters=1000):
  # Get the number of samples (m) and number of features (n)
  m, n = X.shape

  # Initialize weight matrix with shape (n, num_classes)
  theta = np.zeros((n, num_classes))

  # One-hot encode the labels
  y_onehot = one_hot_encode(y, num_classes)

  losses = []

  for _ in tqdm(range(n_iters), desc="Training Multiclass Logistic Regression"):
    # Calculate the logits z
    z = np.dot(X, theta)

    # Get class probabilities using softmax
    y_pred = softmax(z)

    # Compute the gradient of Categorical Cross-Entropy with Softmax
    # ∂J/∂θ = (1/m) * X^T * (y_pred - y_onehot)
    gradient = np.dot(X.T, (y_pred - y_onehot)) / m

    # Update weights
    theta -= lr * gradient

    # Track loss
    loss = categorical_cross_entropy(y_onehot, y_pred)
    losses.append(loss)

  return theta, losses

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
n_splits = 3 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)

In [ ]:
# Calculate average loss across folds
avg_loss = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Logistic Regression Loss', color='purple')
plt.title('Average Logistic Regression Loss Curve (Across 5 Folds)')
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
importances['XGBoost'] = sklearn_models['XGBoost'].feature_importances_
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: